# Deep Learning - Notebook 22
## Modern NLP, Transformers & Hugging Face

Welcome to Module 22.

In Notebook 21, we learned foundational NLP techniques—TF-IDF and dense Word Embeddings (Word2Vec, GloVe). While Word2Vec maps words to static vectors, it suffers from a fatal flaw: **it assigns the exact same vector to a word regardless of context**.

Consider these two sentences:
1. "I went to the **bank** to deposit a check."
2. "I sat on the river **bank** to watch the sunset."

To Word2Vec, the word `"bank"` has the exact same static vector in both sentences.

In 2017, a revolutionary paper titled *"Attention Is All You Need"* introduced the **Transformer Architecture**. Transformers changed Artificial Intelligence forever, giving rise to modern Large Language Models (LLMs) like BERT, GPT-4, Claude, Llama, and Gemini.

In this notebook, we master the Self-Attention mechanism, Transformer architectures, LLM families, and the **Hugging Face Ecosystem** using an anchor scenario:
* **Enterprise AI Document Assistant & LLM Fine-Tuning:** Building an intelligent document processing engine that extracts insights, analyzes contractual risks, summarizes lengthy corporate PDFs, and fine-tunes pre-trained transformer models for custom enterprise classification tasks.

For every topic, we answer three questions:
* What is this in plain English?
* Why are we learning this?
* How is this used in Artificial Intelligence?

---
## Section 1: The Self-Attention Mechanism
### Weighing Contextual Word Relationships

### 1. What is this in plain English?
Before Transformers, older neural networks (RNNs and LSTMs) processed sentences word-by-word sequentially from left to right. By the time an RNN reached word 50 in a sentence, it had already forgotten word 1.

**Self-Attention** processes the entire sentence simultaneously. For every word in a sentence, the Self-Attention mechanism looks at every other word and asks: *"How much attention should I pay to that word to understand this word?"*

Consider the sentence: *"The animal didn't cross the street because **it** was too tired."*
What does the word **"it"** refer to? The animal or the street?
Self-Attention calculates a high mathematical attention weight between **"it"** and **"animal"**, dynamically generating a contextual embedding for "it".

### 2. The Mathematical Equations of Attention:
For every input word token, the network generates three vectors:
* **Query ($Q$):** What am I looking for?
* **Key ($K$):** What information do I contain?
* **Value ($V$):** What actual content do I pass forward?

$$\text{Attention}(Q, K, V) = \text{Softmax}\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

### 3. How is this used in AI?
* The core mathematical engine underlying all modern Large Language Models.

In [1]:
# NUMPY IMPLEMENTATION OF SCALED DOT-PRODUCT SELF-ATTENTION

import numpy as np

def scaled_dot_product_attention(Q, K, V):
    """
    Computes Scaled Dot-Product Attention across input token vectors.
    Q, K, V shape: (seq_len, d_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: Calculate raw attention similarity scores (Q @ K^T)
    scores = Q @ K.T
    
    # Step 2: Scale by square root of d_k to prevent exploding gradients
    scaled_scores = scores / np.sqrt(d_k)
    
    # Step 3: Apply Softmax to convert scores into normalized probability weights (sum = 1.0)
    exp_scores = np.exp(scaled_scores - np.max(scaled_scores, axis=-1, keepdims=True))
    attention_weights = exp_scores / np.sum(exp_scores, axis=-1, keepdims=True)
    
    # Step 4: Multiply attention weights by Value vectors
    output = attention_weights @ V
    
    return output, attention_weights

# SIMULATING A 3-WORD SENTENCE WITH 4-DIMENSIONAL EMBEDDINGS
# Words: ["The", "bank", "river"]
np.random.seed(42)
seq_len = 3
d_k = 4

Q_matrix = np.random.randn(seq_len, d_k)
K_matrix = np.random.randn(seq_len, d_k)
V_matrix = np.random.randn(seq_len, d_k)

contextual_output, weights = scaled_dot_product_attention(Q_matrix, K_matrix, V_matrix)

print("--- Self-Attention Weight Matrix (3x3 Words) ---")
print("Attention Weights:\n", np.round(weights, 4))
print("\nNotice how row 2 ('bank') distributes attention across row 3 ('river')!")
print("\nContextualized Vector Output Shape:", contextual_output.shape)

--- Self-Attention Weight Matrix (3x3 Words) ---
Attention Weights:
 [[0.3929 0.1682 0.439 ]
 [0.2309 0.2834 0.4857]
 [0.2255 0.5587 0.2158]]

Notice how row 2 ('bank') distributes attention across row 3 ('river')!

Contextualized Vector Output Shape: (3, 4)


---
## Section 2: Transformer Architecture & Modern LLM Families

### 1. What is this in plain English?
The original Transformer architecture consists of two main blocks: an **Encoder** (which reads and understands text) and a **Decoder** (which generates new text word-by-word).

Modern Large Language Models have evolved into three distinct families based on this architecture:

1. **Encoder-Only Models (The BERT Family):**
   * *Models:* BERT, RoBERTa, DeBERTa.
   * *How They Work:* Reads text bidirectionally (left-to-right AND right-to-left simultaneously).
   * *Best For:* Text Classification, Sentiment Analysis, Named Entity Recognition, and Vector Search Embeddings.
2. **Decoder-Only Models (The GPT Family):**
   * *Models:* GPT-3.5, GPT-4, Llama 3, Mistral, Claude.
   * *How They Work:* Reads text autoregressively from left to right, predicting the single most likely next word ($N+1$).
   * *Best For:* Generative Writing, Code Generation, Conversational Chatbots, and Reasoning.
3. **Encoder-Decoder Models (The T5 / BART Family):**
   * *Models:* T5 (Text-to-Text Transfer Transformer), BART.
   * *How They Work:* Encoder digests input text; Decoder generates a transformed output text.
   * *Best For:* Document Summarization, Language Translation, and Text Paraphrasing.

---
## Section 3: The Hugging Face Ecosystem
### Tokenizers, Models, and Pipelines

### 1. What is this in plain English?
Building and pre-training a Transformer model from scratch costs millions of dollars in GPU computing.

**Hugging Face** is the "GitHub and App Store of Artificial Intelligence". It provides an open-source library (`transformers`) that allows you to download pre-trained state-of-the-art models in 2 lines of Python code!

### 2. Core Hugging Face Concepts:
* **Tokenizers (`AutoTokenizer`):** Transformers do not split text by spaces. They use **Sub-word Tokenization** (WordPiece or BPE). Unknown words are broken down into sub-word chunks (`"unhelpfulness"` $\to$ `["un", "help", "ful", "ness"]`).
* **AutoModels (`AutoModelForSequenceClassification`):** Pre-trained neural network architectures ready for inference or fine-tuning.
* **Pipelines (`pipeline`):** High-level wrappers that combine tokenization, model inference, and output formatting into one function.

In [3]:
# HUGGING FACE PIPELINE & TOKENIZATION (WITH DEFENSIVE FALLBACK)

try:
    from transformers import pipeline, AutoTokenizer

    # 1. HIGH-LEVEL HUGGING FACE PIPELINE FOR SENTIMENT ANALYSIS
    sentiment_pipeline = pipeline(
        "sentiment-analysis", 
        model="distilbert-base-uncased-finetuned-sst-2-english"
    )

    enterprise_document = "The Q3 revenue figures exceeded expectations, but supply chain costs caused severe margin compression."
    
    result = sentiment_pipeline(enterprise_document)
    print("--- Hugging Face Sentiment Analysis Pipeline Output ---")
    print("Input Text:", enterprise_document)
    print("Sentiment Prediction:", result)


    # 2. SUB-WORD TOKENIZATION DEMONSTRATION
    tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
    
    complex_text = "Transformer architecture enables high-performance fine-tuning."
    tokens = tokenizer.tokenize(complex_text)
    token_ids = tokenizer.encode(complex_text)

    print("\n--- Sub-Word Tokenization Analysis ---")
    print("Original Text: ", complex_text)
    print("Sub-Word Tokens:", tokens)
    print("Numeric Token IDs:", token_ids)

except ImportError:
    print("Note: 'transformers' package not installed. Syntax and architectural concepts demonstrated above.")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1016.61it/s]


--- Hugging Face Sentiment Analysis Pipeline Output ---
Input Text: The Q3 revenue figures exceeded expectations, but supply chain costs caused severe margin compression.
Sentiment Prediction: [{'label': 'NEGATIVE', 'score': 0.9990541338920593}]

--- Sub-Word Tokenization Analysis ---
Original Text:  Transformer architecture enables high-performance fine-tuning.
Sub-Word Tokens: ['transform', '##er', 'architecture', 'enables', 'high', '-', 'performance', 'fine', '-', 'tuning', '.']
Numeric Token IDs: [101, 10938, 2121, 4294, 12939, 2152, 1011, 2836, 2986, 1011, 17372, 1012, 102]


---
## Section 4: Fine-Tuning Pre-Trained Transformers
### Transfer Learning for Enterprise LLMs

### 1. What is this in plain English?
A pre-trained model like DistilBERT already understands general English grammar, vocabulary, and sentence structure from reading Wikipedia.

**Fine-Tuning** is the process of taking a pre-trained Transformer model and training it for 3 to 5 extra epochs on your company's specific dataset (e.g., legal contracts or internal support tickets).

### 2. Parameter-Efficient Fine-Tuning (PEFT / LoRA):
Fine-tuning a 70-billion parameter model completely is extremely expensive. Modern AI uses **LoRA (Low-Rank Adaptation)**, which freezes 99% of the original model weights and injects tiny, trainable low-rank rank-decomposition matrices into the self-attention layers!

In [4]:
# FINE-TUNING TRANSFORMER ARCHITECTURE SETUP IN PYTORCH / TRANSFORMERS

try:
    import torch
    from transformers import AutoModelForSequenceClassification, AutoTokenizer

    # Load pre-trained BERT classification architecture with 2 custom enterprise classes
    model_name = "distilbert-base-uncased"
    
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    # Sample Enterprise Document Batch
    documents = [
        "Contract Clause 4A: Vendor guarantees 99.9% system uptime.",
        "Warning: Default on payment schedule triggers immediate account termination."
    ]

    # Tokenize batch for PyTorch model input
    inputs = tokenizer(documents, padding=True, truncation=True, return_tensors="pt")

    # Forward pass through pre-trained Transformer
    outputs = model(**inputs)
    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)

    print("--- PyTorch Transformer Forward Pass Completed ---")
    print("Inputs Tensor Keys: ", list(inputs.keys()))
    print("Raw Logits Shape:   ", logits.shape)
    print("Class Predictions:  ", predictions.numpy())

except ImportError:
    print("Note: 'transformers' or 'torch' package not installed. Fine-tuning setup demonstrated above.")

C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\moham\.cache\huggingface\hub. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
C:\Users\moham\AppData\Roaming\Python\Python314\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cac

--- PyTorch Transformer Forward Pass Completed ---
Inputs Tensor Keys:  ['input_ids', 'token_type_ids', 'attention_mask']
Raw Logits Shape:    torch.Size([2, 2])
Class Predictions:   [1 1]


---
## Hands-on Workshop: Modern Transformers & Hugging Face

Complete these four practical challenges to build your LLM engineering capabilities.

### Challenge 1: Scaled Dot-Product Attention Calculation (Junior Level)

**Scenario:** Manually calculate attention weights for a 2-token sentence using NumPy.

**Task Requirements:**
1. Given Query matrix `Q = np.array([[1.0, 0.0], [0.0, 1.0]])` and Key matrix `K = np.array([[1.0, 0.0], [0.0, 1.0]])`.
2. Compute raw scores `scores = Q @ K.T`.
3. Scale by `np.sqrt(2.0)`.
4. Apply Softmax to obtain attention weights and print.

In [4]:
# --- CHALLENGE 1: YOUR WORKSPACE ---




### Challenge 2: Hugging Face Summarization Pipeline (Mid Level)

**Scenario:** Automate document summarization for long executive PDF reports.

**Task Requirements:**
1. Use `transformers.pipeline("summarization")`.
2. Pass a long paragraph of corporate text.
3. Generate and print the concise summary output.

In [3]:
# --- CHALLENGE 2: YOUR WORKSPACE ---




### Challenge 3: Tokenizer Sub-word Decoding (Senior Level)

**Scenario:** Inspect how Hugging Face tokenizers handle domain-specific jargon.

**Task Requirements:**
1. Load `AutoTokenizer.from_pretrained("bert-base-uncased")`.
2. Tokenize domain term `"hyperparameter optimization pipeline"`.
3. Print the resulting sub-word token strings.

In [1]:
# --- CHALLENGE 3: YOUR WORKSPACE ---




### Challenge 4: Zero-Shot Document Classification Pipeline (Architect Level)

**Scenario:** Classify legal and compliance documents without any fine-tuning using Zero-Shot Transformer pipelines.

**Task Requirements:**
1. Initialize `pipeline("zero-shot-classification", model="facebook/bart-large-mnli")`.
2. Pass document `"Vendor agrees to maintain SOC2 security compliance and encrypt customer data at rest."`.
3. Pass candidate labels `["Security Compliance", "Billing Issue", "Marketing Strategy"]`.
4. Print predicted labels and probability scores.

In [2]:
# --- CHALLENGE 4: YOUR WORKSPACE ---




# Session Summary

Congratulations. You have completed Notebook 22.

Let us review what you now understand:
1. **Self-Attention Mechanism:** You know how Transformers calculate $Q K^T V$ attention matrices to capture context across sentences simultaneously.
2. **LLM Architectures:** You understand the functional differences between Encoder-Only (BERT - classification), Decoder-Only (GPT - generation), and Encoder-Decoder (T5 - summarization).
3. **Hugging Face Ecosystem:** You know how to use tokenizers, pipelines, and pre-trained auto-models.
4. **Fine-Tuning:** You understand how transfer learning and parameter-efficient fine-tuning (LoRA) adapt base LLMs for specialized enterprise document processing.

In **Notebook 23**, we conclude the Deep Learning curriculum with **Advanced Deep Learning & Frontier Models**—learning Autoencoders (VAEs), Generative Adversarial Networks (GANs), Reinforcement Learning (Q-Learning), and Graph Neural Networks (GNNs)!